In [1]:
import math
import sys
from pathlib import Path

REPO = Path.cwd().resolve().parent  # this notebook lives in demos/
sys.path.insert(0, str(REPO))

import os
os.environ["DISPLAY"] = ":0"

from launcher import start_giskard_server, start_isaac_sim, start_rviz, stop

ROBOT = "panda"  # in the default scene, the apartment

# Where the arm is bolted down, in the map frame. Measured in apartmentICRA.usda
# as (7.14, -5.30, 0.933) and composed with the prim placement (-6.0, 5.0) in x/y
# only -- see the table above. Yaw pi faces it along map -x, out over the table.
SPAWN_POSITION = (1.84, -0.30, 0.633)
SPAWN_YAW = math.pi

# rviz_proc = start_rviz()
# sim_proc = start_isaac_sim(robot=ROBOT, spawn_position=SPAWN_POSITION,
#                            spawn_yaw=SPAWN_YAW)
# giskard_proc = start_giskard_server(robot=ROBOT, spawn_position=SPAWN_POSITION,
#                                     spawn_yaw=SPAWN_YAW)

In [2]:
import threading
import numpy as np

import nest_asyncio
import rclpy
from rclpy.executors import MultiThreadedExecutor

nest_asyncio.apply()  # CRAM's REAL execution calls GiskardWrapper.execute,
                      # which run_until_completes inside the already-running kernel loop.

from coraplex.datastructures.dataclasses import Context
from semantic_digital_twin.adapters.ros.world_fetcher import fetch_world_from_service
from semantic_digital_twin.adapters.ros.world_synchronizer import WorldSynchronizer

from cram_vrb_lab.robots.panda.motions import PANDA_MOTION_MAPPINGS
from cram_vrb_lab.robots.panda.semantic_model import Panda

if not rclpy.ok():
    rclpy.init()
node = rclpy.create_node('cram_panda_node')
executor = MultiThreadedExecutor()
executor.add_node(node)
threading.Thread(target=executor.spin, daemon=True, name='rclpy-executor').start()

world = fetch_world_from_service(node=node, timeout_seconds=300)
WorldSynchronizer(_world=world, node=node)

robot = world.get_semantic_annotations_by_type(Panda)
robot = robot[0] if robot else Panda.from_world(world)

context = Context(
    world=world,
    robot=robot,
    ros_node=node,
    evaluate_conditions=False,
    alternative_motion_mappings=PANDA_MOTION_MAPPINGS,
)
print('connected, robot:', type(robot).__name__)

connected, robot: Panda


## A small run helper

`with real_robot(...)` sets the execution type to REAL, so `plan.perform()`
builds each action's giskard motion and streams it to the running server.

In [4]:
from coraplex.datastructures.enums import Arms
from coraplex.execution_environment import real_robot
from coraplex.plans.factories import execute_single, sequential
from coraplex.view_manager import ViewManager


def run_plan(plan, collision_avoidance=True):
    """Perform a CRAM plan on the real (sim) robot via giskard."""
    with real_robot(collision_avoidance=collision_avoidance):
        plan.perform()
    print('done')


def tool_position():
    """The gripper's tool frame, in map."""
    tool = ViewManager.get_end_effector_view(Arms.LEFT, robot).tool_frame
    return np.asarray(tool.global_pose.to_np())[:3, 3].ravel()

def tool_axis():
    """Where the gripper points: the tool frame's z-axis, in map."""
    tool = ViewManager.get_end_effector_view(Arms.LEFT, robot).tool_frame
    return np.asarray(tool.global_pose.to_np())[:3, 2].ravel()

def body_position(name):
    """Any body of the flat, in map -- straight out of the MJCF twin."""
    return np.asarray(world.get_body_by_name(name).global_pose.to_np())[:3, 3].ravel()

In [7]:
from semantic_digital_twin.semantic_annotations.semantic_annotations import (
    Drawer,
    Handle,
)

drawer_body = world.get_body_by_name(f"cabinet10_drawer_middle")
handle_body = world.get_body_by_name(f"handle_cab10_m")

if not world.get_semantic_annotations_by_type(Drawer):
    with world.modify_world():
        world.add_semantic_annotation_recursively(
            Drawer(root=drawer_body, handle=Handle(root=handle_body))
        )
print("drawer annotated:", drawer_body.name, "with handle", handle_body.name)

drawer annotated: cabinet10_drawer_middle with handle handle_cab10_m


In [5]:
from coraplex.datastructures.enums import Arms
from coraplex.robot_plans.actions.core.robot_body import ParkArmsAction, SetGripperAction
from semantic_digital_twin.datastructures.definitions import GripperState

from cram_vrb_lab.robots.panda.joints import FINGER_JOINTS, gripper_pad_gap

run_plan(execute_single(ParkArmsAction(Arms.LEFT), context=context))
run_plan(execute_single(SetGripperAction(Arms.LEFT, GripperState.OPEN), context=context))

[INFO] [1786110890.622427800] [cram_panda_node]: giskard/command Goal #0 accepted
[INFO] [1786110894.349960038] [cram_panda_node]: giskard/command Goal #0 result received


done


[INFO] [1786110894.747246800] [cram_panda_node]: giskard/command Goal #0 accepted


done


[INFO] [1786110895.950147415] [cram_panda_node]: giskard/command Goal #0 result received


In [11]:
run_plan(execute_single(SetGripperAction(Arms.LEFT, GripperState.CLOSE), context=context))

[INFO] [1786111085.390536619] [cram_panda_node]: giskard/command Goal #0 accepted
[INFO] [1786111085.453894151] [cram_panda_node]: giskard/command Goal #0 result received


done


[INFO] [1786111089.453095009] [cram_panda_node]: giskard/command Goal #0 result received


In [ ]:
run_plan(execute_single(SetGripperAction(Arms.LEFT, GripperState.OPEN), context=context))


In [ ]:
import math
from time import sleep

from coraplex.robot_plans.motions.gripper import MoveToolCenterPointMotion
from semantic_digital_twin.spatial_types import Point3, Pose, Quaternion, RotationMatrix

Z_ALONG_HANDLE_X = RotationMatrix.from_quaternion(
    Quaternion(0.0, math.sqrt(2) / 2, 0.0, math.sqrt(2) / 2)
)

grasp_ori = (
    handle_body.global_pose.to_rotation_matrix() @ Z_ALONG_HANDLE_X
).to_quaternion()


WAYPOINTS = [(1.6 - i *0.02, -0.342356, 0.7251) for i in range(1)]

for point in WAYPOINTS:
    run_plan(execute_single(
        MoveToolCenterPointMotion(
            Pose(
                position=Point3.from_iterable(point),
                orientation=grasp_ori,
                reference_frame=world.root,
            ),
            Arms.LEFT,
        ),
        context=context,
    ))
    reached, pointing = tool_position(), tool_axis()
    print(f'asked {np.round(point, 3)} '
          f'reached {np.round(reached, 3)}  '
          f'err {np.linalg.norm(reached - np.array(point)) * 100:.1f} cm  '
          f'pointing {np.round(pointing, 3)}')
    # sleep(0.3)

In [10]:

from coraplex.robot_plans.actions.core.container import OpenAction, CloseAction

run_plan(
    execute_single(OpenAction(handle_body, Arms.LEFT), context=context),
    collision_avoidance=False,
)
print("drawer joint:", world.get_connection_by_name("cabinet9_drawer_middle").position)

[INFO] [1786111020.946278813] [cram_panda_node]: giskard/command Goal #0 accepted
[INFO] [1786111021.003035273] [cram_panda_node]: giskard/command Goal #0 result received


KeyboardInterrupt: 

In [ ]:
run_plan(
    execute_single(CloseAction(handle_body, Arms.LEFT), context=context),
    collision_avoidance=True,
)

## 1. Put the props into the digital twin

Isaac already has the cube as a physics body. CRAM plans against
the twin, so it has to exist there too — `add_props_to_twin` builds it from
`panda_layout_at(SPAWN_POSITION, SPAWN_YAW)`, the same function and the same
spawn pose the sim built its cube from, so the two cannot describe different
cubes. The change goes out over
`/world_sync`, so the giskard server's copy of the world learns about it. The
apartment is already in that world: the giskard server merges it in at startup,
so collision avoidance covers the room as well.

With no pedestals in the layout only the cube is added. Whatever it rests on
belongs to the scene — and if that table is not in the apartment URDF either,
giskard plans as though nothing were underneath it.

`sync_cube_from_sim` then snaps the twin's cube onto the pose Isaac's physics
actually settled it at — a one-shot perception stand-in, and the only channel
through which the twin ever learns the cube is not where CRAM assumed.

In [ ]:
import numpy as np

from cram_vrb_lab.scenes.props import constants as props
from cram_vrb_lab.scenes.props.constants import panda_layout_at
from cram_vrb_lab.scenes.props.twin_props import (
    CubePoseSensor,
    add_props_to_twin,
    sync_cube_from_sim,
)

# Where the sim put the cube: in front of the arm, on the surface it is mounted
# on. Derived from the spawn pose above exactly as the sim derives it.
PANDA_LAYOUT = panda_layout_at(SPAWN_POSITION, SPAWN_YAW)

cube = add_props_to_twin(world, layout=PANDA_LAYOUT)
cube_sensor = CubePoseSensor(node)

print('twin cube:', np.round(np.asarray(cube.global_pose.to_np())[:3, 3], 3))
print('sim cube: ', np.round(sync_cube_from_sim(world, cube, cube_sensor), 3))

### Believed vs. actual

The one measurement this notebook is built around. CRAM's `AttachNode` moves the
cube along with the gripper *in the twin* the moment the close-gripper motion
finishes — whether or not the physical fingers caught anything. So the twin
always reports a successful grasp. Isaac does not.

While the cube is held, a gap of a centimetre or two is normal (the twin freezes
it at the tool frame; the real cube sits wherever the fingers gripped it). A gap
that keeps growing means the cube was left behind or dropped.

In [ ]:
def cube_status(label=''):
    """Print where CRAM believes the cube is vs where Isaac's physics has it."""
    believed = np.asarray(cube.global_pose.to_np())[:3, 3].ravel()
    actual = np.array(cube_sensor.position())
    print(f'{label:14s} twin {np.round(believed, 3)}  '
          f'sim {np.round(actual, 3)}  gap {np.linalg.norm(believed - actual):.3f} m')
    return believed, actual


cube_status('start')

## 2. Park the arm and open the hand

`ParkArmsAction` looks up the Panda's park configuration (Franka's own "ready"
pose) and `SetGripperAction` its open finger travel, both from the semantic
model. Unlike the Stretch, no widening is needed: the hand's `GripperState.OPEN`
is 0.038 m of travel per finger, i.e. **0.076 m between the pads** against a
0.05 m cube — 1.3 cm of clearance either side, which is how much approach error
the grasp tolerates before a finger knocks the cube off its stand.

In [ ]:
from coraplex.datastructures.enums import Arms
from coraplex.robot_plans.actions.core.robot_body import ParkArmsAction, SetGripperAction
from semantic_digital_twin.datastructures.definitions import GripperState

from cram_vrb_lab.robots.panda.joints import FINGER_JOINTS, gripper_pad_gap

# run_plan(execute_single(ParkArmsAction(Arms.LEFT), context=context))
# run_plan(execute_single(SetGripperAction(Arms.LEFT, GripperState.OPEN), context=context))

# travel = world.get_connection_by_name(FINGER_JOINTS[0]).position
# print(f'fingers at {travel:.3f} m travel = {gripper_pad_gap(travel):.3f} m between '
#       f'the pads (needs to clear the {props.CUBE_SIZE:.3f} m cube)')

`Arms.LEFT` is not a claim about handedness — the Panda has one arm, and CRAM's
`ViewManager` hands back the only arm there is whatever you ask for. It just has
to be *some* member of the enum.

## 3. Pick the cube up and put it down

Both halves run as **one plan**, and that is not a stylistic choice — it is the
only way to make the place go top-down.

**`PlaceAction` takes no grasp.** Its fields are just `object_designator`,
`target_location`, `arm`. It recovers the orientation by looking for a
`PickUpAction` *earlier in the same plan* and reusing that pick's
`grasp_description`
(`coraplex/robot_plans/actions/core/placing.py`), falling back to
`FRONT` + `NoAlignment` when it finds none. Run the pick and the place as two
separate `execute_single(...)` plans and the place always gets that fallback —
whose three rotation factors are all identity, so the target tool rotation
reduces to exactly the gripper's `front_facing_orientation`. For the Panda that
is the 90° rotation about y, i.e. hand z along map **+x**: a horizontal approach,
with a horizontal retract that drags the hand across the table.

`PickAndPlaceAction` ("without moving the base") keeps them in one plan, so the
place inherits `FRONT` + `TOP`. It expands to:

    ParkArms → PickUp(grasp) → ParkArms → Place → ParkArms

With `FRONT` + `TOP` every tool pose is vertical, in map coordinates:

| | position (map) | hand points |
|---|---|---|
| pick pre-grasp | (0.69, −0.10, 1.033) | down |
| pick grasp | (0.69, −0.10, 0.958) | down |
| pick lift | (0.69, −0.10, 1.008) | down |
| place descend | (0.69, −0.50, 1.008) | down |
| place release | (0.69, −0.50, 0.958) | down |
| place retract | (0.69, −0.50, 1.033) | down |

(Nominal, from `surface_z = 0.933`; the grasp is re-planned against the cube's
settled pose, so the real numbers follow wherever the table actually is. The
7.5 cm of approach clearance is the cube's half-height plus
`GraspDescription.manipulation_offset`.)

`collision_avoidance=False` for the same reason as before — the hand has to get
*inside* the avoidance margin of the cube and the table, which is exactly what the
margin forbids. Note that one `real_robot(...)` context now wraps the whole
composite, so the parks and the transfer run without external collision avoidance
too, not just the two reaches.

**Why the close-gripper step needs a Panda-specific motion.** CRAM turns a
gripper state into a joint-position goal on the fingers, and under closed-loop
control that goal is checked against the *measured* finger positions. A hand
closing on a rigid object never reaches them — the cube stops the fingers 2.5 cm
short of closed — so the default 1 cm tolerance is never met and the grasp hangs
right there, holding the cube, looking for all the world like it worked.
`PandaMoveGripper` (in `cram_vrb_lab/robots/panda/motions.py`) gives *closing*
a 3 cm tolerance. Opening keeps the tight one; nothing obstructs it, and the
place's release needs it to open fully.

Ending the close is not the same as letting go: the fingers keep driving towards a
target that now lies inside the cube, so they go on squeezing through the lift and
the transfer.

We re-sync from the sim first, so the grasp is planned against where the cube
really is rather than where it was spawned.

In [ ]:
from coraplex.datastructures.enums import ApproachDirection, VerticalAlignment
from coraplex.datastructures.grasp import GraspDescription
from coraplex.robot_plans.actions.composite.transporting import PickAndPlaceAction
from coraplex.view_manager import ViewManager
from semantic_digital_twin.spatial_types import Point3
from semantic_digital_twin.spatial_types.spatial_types import Pose

grasp = GraspDescription(
    ApproachDirection.FRONT,   # with TOP below: approach along map -z
    VerticalAlignment.TOP,
    ViewManager.get_end_effector_view(Arms.LEFT, robot),
)
place_target = Pose(
    Point3.from_iterable((0.73, -0.4, 1.2)),
    reference_frame=world.root,
)

sync_cube_from_sim(world, cube, cube_sensor)
run_plan(
    execute_single(
        # One plan, so PlaceAction can find this grasp on the PickUpAction node.
        PickAndPlaceAction(cube, place_target, Arms.LEFT, grasp_description=grasp),
        context=context,
    ),
    collision_avoidance=False,
)
cube_status('after place')

### What the merge costs

Running the pick and the place as one plan is what buys the top-down place, but it
takes away the measurement in between. Previously `cube_status('after grasp')` ran
while the cube was in the air, which is where a grasp that closes on nothing shows
itself immediately. Now a cube that is lifted and then dropped mid-transfer looks
the same from the outside as one that was never picked up — both end with the cube
somewhere it should not be, and only the verdict below reports it. Watch the Isaac
viewport for the difference, or re-add a `cube_status` call by splitting the
composite back into `sequential([PickUpAction(...), PlaceAction(...)])` and
accepting that you cannot interleave Python between plan nodes.

The composite already ends with a park, so no separate `ParkArmsAction` is needed
after it.

## 4. Verdict

The cube's true resting position against where it was asked to go. Both parts
matter: the height says it is still on the table rather than on the floor, and the
x/y error says it is at the *right* spot on it — a cube that was never picked up
would score a perfect height and be 0.4 m out in y, which is exactly the failure
the merged plan can no longer catch mid-flight.

In [ ]:
_, actual = cube_status('final')
target = np.array(PANDA_LAYOUT.cube_target_position)
error = actual - target
placed = np.linalg.norm(error[:2]) < 0.05 and abs(error[2]) < 0.02
print(f'target {np.round(target, 3)}')
print(f'error  {np.round(error, 3)}   |xy| {np.linalg.norm(error[:2]):.3f} m')
print('PLACED on the table' if placed else 'NOT placed -- see cube_status above')

## Shutdown

In [ ]:
stop()  # stops the isaac sim + giskard server + rviz started above